In [1]:
import pandas as pd
df=pd.read_csv("/content/100_Unique_QA_Dataset.csv")
print(df)

                                             question        answer
0                      What is the capital of France?         Paris
1                     What is the capital of Germany?        Berlin
2                  Who wrote 'To Kill a Mockingbird'?    Harper-Lee
3     What is the largest planet in our solar system?       Jupiter
4      What is the boiling point of water in Celsius?           100
..                                                ...           ...
85                  Who directed the movie 'Titanic'?  JamesCameron
86  Which superhero is also known as the Dark Knight?        Batman
87                     What is the capital of Brazil?      Brasilia
88        Which fruit is known as the king of fruits?         Mango
89       Which country is known for the Eiffel Tower?        France

[90 rows x 2 columns]


In [2]:
# tokens=ise the text
def tokenize(text):
  text=text.lower()
  text=text.replace(".","")
  text=text.replace(",","")
  text=text.replace("?","")
  text=text.replace("'","")
  return text.split()

In [3]:
tokenize("what is sugar?")

['what', 'is', 'sugar']

In [4]:
# vocabulary
vocab = {'<UNK>':0}

In [9]:
# vocabulary
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)


df.apply(build_vocab, axis=1)
print(len(vocab))


324


In [12]:
#numerical indices
def text_to_indices(text, vocab):
  indexed_text=[]

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

text_to_indices("what is GDP?", vocab)

[1, 2, 0]

In [16]:
import torch
from torch.utils.data import DataLoader, Dataset

class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)


dataset=QADataset(df, vocab)
print(dataset[10])


(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))


In [29]:
dataloader=DataLoader(dataset, batch_size=1, shuffle=True)
for question, answer in dataloader:
  print(question, answer[0])

tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([121])
tensor([[ 10, 308,   3, 309, 310]]) tensor([311])
tensor([[ 42, 107,   2, 108,  19, 109]]) tensor([110])
tensor([[  1,   2,   3,   4,   5, 206]]) tensor([207])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([36])
tensor([[1, 2, 3, 4, 5, 6]]) tensor([7])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([295])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([287])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([72])
tensor([[ 42, 101,   2,   3,  17]]) tensor([102])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([254])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([53])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([317])
tensor([[10, 75, 76]]) tensor([77])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177,

In [48]:
import torch.nn as nn
class RNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

learning_rate=0.001
epochs=100

In [49]:
model=RNN(len(vocab))
#model.to()

In [50]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=learning_rate)

In [51]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.409760
Epoch: 2, Loss: 456.195519
Epoch: 3, Loss: 376.565775
Epoch: 4, Loss: 313.818233
Epoch: 5, Loss: 260.771246
Epoch: 6, Loss: 211.824514
Epoch: 7, Loss: 168.679270
Epoch: 8, Loss: 131.111940
Epoch: 9, Loss: 100.721140
Epoch: 10, Loss: 77.258419
Epoch: 11, Loss: 59.950345
Epoch: 12, Loss: 46.835673
Epoch: 13, Loss: 37.837661
Epoch: 14, Loss: 30.419120
Epoch: 15, Loss: 25.032033
Epoch: 16, Loss: 20.914500
Epoch: 17, Loss: 18.001117
Epoch: 18, Loss: 14.887933
Epoch: 19, Loss: 12.803112
Epoch: 20, Loss: 11.077283
Epoch: 21, Loss: 9.718434
Epoch: 22, Loss: 8.491631
Epoch: 23, Loss: 7.528120
Epoch: 24, Loss: 6.706659
Epoch: 25, Loss: 6.036947
Epoch: 26, Loss: 5.442906
Epoch: 27, Loss: 4.914708
Epoch: 28, Loss: 4.471637
Epoch: 29, Loss: 4.085822
Epoch: 30, Loss: 3.758010
Epoch: 31, Loss: 3.438731
Epoch: 32, Loss: 3.160040
Epoch: 33, Loss: 2.931557
Epoch: 34, Loss: 2.712286
Epoch: 35, Loss: 2.507605
Epoch: 36, Loss: 2.337991
Epoch: 37, Loss: 2.170577
Epoch: 38, Loss: 2

In [52]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [53]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [54]:
predict(model," What is the capital of Germany?")

berlin


In [55]:
predict(model,"Which superhero is also known as the Dark Knight?")

batman
